# Production Forecasting Demo

This notebook is a Colab-ready version of the production forecasting workshop demo.

## What it does
- loads the synthetic production and event data
- builds a simple decline-style baseline and a machine-learning model
- evaluates both on a 6-month holdout window
- creates a 6-month forward forecast for each well
- flags likely underperformers and likely operational drivers

## Colab note
If the sample data is not already available, the notebook will prompt you to upload these files:
- `well_monthly_production.csv`
- `well_events.csv`
- `data_dictionary.md`


In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid")

IN_COLAB = "google.colab" in sys.modules
REQUIRED_FILES = [
    "well_monthly_production.csv",
    "well_events.csv",
    "data_dictionary.md",
]


def locate_data_dir() -> Path:
    cwd = Path.cwd()
    candidate_dirs = [
        cwd / "training" / "sample-data" / "production_forecasting",
        cwd / "sample-data" / "production_forecasting",
        cwd / "production_forecasting",
        cwd.parent / "sample-data" / "production_forecasting",
    ]

    for path in candidate_dirs:
        if all((path / name).exists() for name in REQUIRED_FILES):
            return path

    if IN_COLAB:
        from google.colab import files

        print("Upload the production forecasting demo files.")
        uploaded = files.upload()
        upload_dir = cwd / "production_forecasting"
        upload_dir.mkdir(parents=True, exist_ok=True)

        for name, content in uploaded.items():
            file_name = Path(name).name
            if file_name in REQUIRED_FILES:
                (upload_dir / file_name).write_bytes(content)

        missing = [name for name in REQUIRED_FILES if not (upload_dir / name).exists()]
        if not missing:
            return upload_dir

        raise FileNotFoundError(f"Missing uploaded files: {missing}")

    raise FileNotFoundError("Could not locate production_forecasting sample data.")


In [ ]:
data_dir = locate_data_dir()

production = pd.read_csv(data_dir / "well_monthly_production.csv", parse_dates=["date"])
events = pd.read_csv(data_dir / "well_events.csv", parse_dates=["event_date"])
dictionary_text = Path(data_dir / "data_dictionary.md").read_text()

print(f"Loaded sample data from: {data_dir}")
display(Markdown("## Data Dictionary"))
display(Markdown(dictionary_text))

summary_df = pd.DataFrame(
    [
        {"metric": "Rows", "value": len(production)},
        {"metric": "Wells", "value": production["well_id"].nunique()},
        {"metric": "Pads", "value": production["pad"].nunique()},
        {"metric": "Date range", "value": f"{production['date'].min().date()} to {production['date'].max().date()}"},
        {"metric": "Event records", "value": len(events)},
    ]
)

display(summary_df)
display(production.head())
display(events.head())


## Business Framing

The engineer wants something practical and teachable:
- forecast near-term oil production at the well level
- separate normal decline from operational underperformance
- surface likely drivers like downtime, water cut, and recent interventions
- package the workflow into something reusable

For a 1-hour workshop, a sensible approach is:
1. create a simple decline-style baseline
2. train a machine-learning model for next-month oil rate
3. compare them on a recent holdout window
4. roll the latest well state forward to build a 6-month forecast


In [ ]:
df = production.sort_values(["well_id", "date"]).copy()
df["event_type"] = df["event_type"].fillna("none")

events = events.sort_values(["well_id", "event_date"]).copy()
event_counts = (
    events.groupby("well_id")
    .size()
    .rename("total_event_count")
    .reset_index()
)
recent_events = (
    events.loc[events["event_date"] >= (df["date"].max() - pd.DateOffset(months=12))]
    .groupby("well_id")
    .size()
    .rename("recent_event_count_12m")
    .reset_index()
)

df = df.merge(event_counts, on="well_id", how="left")
df = df.merge(recent_events, on="well_id", how="left")
df[["total_event_count", "recent_event_count_12m"]] = df[["total_event_count", "recent_event_count_12m"]].fillna(0)

df["oil_rate_1m_lag"] = df.groupby("well_id")["oil_rate_bopd"].shift(1)
df["oil_rate_3m_avg"] = (
    df.groupby("well_id")["oil_rate_bopd"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)
df["oil_rate_6m_avg"] = (
    df.groupby("well_id")["oil_rate_bopd"]
    .transform(lambda s: s.shift(1).rolling(6, min_periods=1).mean())
)
df["oil_rate_3m_decline_pct"] = 100.0 * (df["oil_rate_bopd"] - df["oil_rate_3m_avg"]) / df["oil_rate_3m_avg"]
df["month_number"] = df["date"].dt.month
df["recent_event_flag"] = (df["event_flag"].fillna(0) > 0).astype(int)

usable_df = df.dropna(subset=["forecast_target_next_month_bopd"]).copy()
cutoff_date = usable_df["date"].max() - pd.DateOffset(months=5)

train_df = usable_df.loc[usable_df["date"] < cutoff_date].copy()
test_df = usable_df.loc[usable_df["date"] >= cutoff_date].copy()

display(pd.DataFrame([
    {"split": "train", "rows": len(train_df), "date_range": f"{train_df['date'].min().date()} to {train_df['date'].max().date()}"},
    {"split": "test", "rows": len(test_df), "date_range": f"{test_df['date'].min().date()} to {test_df['date'].max().date()}"},
]))

feature_columns = [
    "pad",
    "oil_rate_bopd",
    "oil_rate_1m_lag",
    "oil_rate_3m_avg",
    "oil_rate_6m_avg",
    "oil_rate_3m_decline_pct",
    "gas_rate_mscfd",
    "water_cut_pct",
    "uptime_pct",
    "choke_size_64in",
    "tubing_pressure_psi",
    "casing_pressure_psi",
    "gor_scf_bbl",
    "bhp_proxy_psi",
    "artificial_lift_type",
    "reservoir_zone",
    "well_age_months",
    "lateral_length_ft",
    "completion_stage_count",
    "recent_event_flag",
    "event_type",
    "total_event_count",
    "recent_event_count_12m",
    "month_number",
]

target_column = "forecast_target_next_month_bopd"


In [ ]:
train_ratio = (train_df[target_column] / train_df["oil_rate_bopd"]).replace([np.inf, -np.inf], np.nan)
median_decline_ratio = float(train_ratio.dropna().median())

test_df = test_df.copy()
test_df["baseline_pred_bopd"] = test_df["oil_rate_bopd"] * median_decline_ratio

numeric_features = [
    col for col in feature_columns
    if col not in ["pad", "artificial_lift_type", "reservoir_zone", "event_type"]
]
categorical_features = ["pad", "artificial_lift_type", "reservoir_zone", "event_type"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

model.fit(train_df[feature_columns], train_df[target_column])
test_df["ml_pred_bopd"] = model.predict(test_df[feature_columns])

metrics_df = pd.DataFrame([
    {
        "model": "Decline-style baseline",
        "MAE": round(mean_absolute_error(test_df[target_column], test_df["baseline_pred_bopd"]), 2),
        "RMSE": round(mean_squared_error(test_df[target_column], test_df["baseline_pred_bopd"], squared=False), 2),
        "R2": round(r2_score(test_df[target_column], test_df["baseline_pred_bopd"]), 3),
    },
    {
        "model": "Random forest",
        "MAE": round(mean_absolute_error(test_df[target_column], test_df["ml_pred_bopd"]), 2),
        "RMSE": round(mean_squared_error(test_df[target_column], test_df["ml_pred_bopd"], squared=False), 2),
        "R2": round(r2_score(test_df[target_column], test_df["ml_pred_bopd"]), 3),
    },
])

display(metrics_df)
display(test_df[["well_id", "date", "oil_rate_bopd", target_column, "baseline_pred_bopd", "ml_pred_bopd"]].head(12))


In [ ]:
plot_sample = test_df.copy()
plot_sample["abs_error_baseline"] = (plot_sample[target_column] - plot_sample["baseline_pred_bopd"]).abs()
plot_sample["abs_error_ml"] = (plot_sample[target_column] - plot_sample["ml_pred_bopd"]).abs()

best_wells = (
    plot_sample.groupby("well_id")["abs_error_ml"]
    .mean()
    .sort_values()
    .head(4)
    .index
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.scatterplot(
    data=plot_sample,
    x=target_column,
    y="ml_pred_bopd",
    hue="pad",
    ax=axes[0],
)
axes[0].plot(
    [plot_sample[target_column].min(), plot_sample[target_column].max()],
    [plot_sample[target_column].min(), plot_sample[target_column].max()],
    linestyle="--",
    color="black",
)
axes[0].set_title("Actual vs ML Prediction")
axes[0].set_xlabel("Actual next-month oil rate")
axes[0].set_ylabel("Predicted next-month oil rate")

for well_id in best_wells:
    well_plot = plot_sample.loc[plot_sample["well_id"] == well_id].sort_values("date")
    axes[1].plot(well_plot["date"], well_plot[target_column], marker="o", label=f"{well_id} actual")
    axes[1].plot(well_plot["date"], well_plot["ml_pred_bopd"], linestyle="--", label=f"{well_id} pred")

axes[1].set_title("Sample Holdout Forecast Traces")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Oil rate bopd")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(ncol=2, fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
latest_df = df.sort_values(["well_id", "date"]).groupby("well_id").tail(1).copy()


def prepare_future_row(row, history_window, step):
    future = row.copy()
    future["date"] = row["date"] + pd.DateOffset(months=1)
    future["month_number"] = future["date"].month
    future["well_age_months"] = row["well_age_months"] + 1
    future["recent_event_flag"] = 0
    future["event_flag"] = 0
    future["event_type"] = "none"
    future["recent_event_count_12m"] = 0

    current_rate = history_window[-1]
    future["oil_rate_bopd"] = current_rate
    future["oil_rate_1m_lag"] = current_rate
    future["oil_rate_3m_avg"] = float(np.mean(history_window[-3:]))
    future["oil_rate_6m_avg"] = float(np.mean(history_window[-6:]))
    avg_3m = future["oil_rate_3m_avg"]
    future["oil_rate_3m_decline_pct"] = 100.0 * (current_rate - avg_3m) / avg_3m if avg_3m else 0.0

    future["water_cut_pct"] = min(float(row["water_cut_pct"]) + 0.5 * step if pd.notna(row["water_cut_pct"]) else 35.0 + 0.5 * step, 98.0)
    future["uptime_pct"] = float(row["uptime_pct"]) if pd.notna(row["uptime_pct"]) else 94.0
    gor_value = float(row["gor_scf_bbl"]) if pd.notna(row["gor_scf_bbl"]) else 1400.0
    future["gas_rate_mscfd"] = current_rate * gor_value / 1000.0

    return future


forecast_rows = []
for _, latest_row in latest_df.iterrows():
    history = (
        df.loc[df["well_id"] == latest_row["well_id"], "oil_rate_bopd"]
        .tail(6)
        .tolist()
    )
    rolling_row = latest_row.copy()

    for step in range(1, 7):
        future_row = prepare_future_row(rolling_row, history, step)
        predicted_rate = float(model.predict(pd.DataFrame([future_row[feature_columns]]))[0])
        predicted_rate = max(predicted_rate, 0.0)

        future_row["forecast_month"] = step
        future_row["forecast_oil_rate_bopd"] = predicted_rate
        forecast_rows.append(future_row.copy())

        history.append(predicted_rate)
        history = history[-6:]
        rolling_row = future_row.copy()
        rolling_row["oil_rate_bopd"] = predicted_rate

forecast_df = pd.DataFrame(forecast_rows)

forecast_summary = (
    forecast_df.groupby("well_id")
    .agg(
        pad=("pad", "first"),
        month_1_forecast_bopd=("forecast_oil_rate_bopd", "first"),
        avg_6m_forecast_bopd=("forecast_oil_rate_bopd", "mean"),
        end_6m_forecast_bopd=("forecast_oil_rate_bopd", "last"),
    )
    .reset_index()
)

latest_snapshot = latest_df[["well_id", "pad", "oil_rate_bopd", "water_cut_pct", "uptime_pct", "recent_event_count_12m"]].copy()
latest_snapshot = latest_snapshot.rename(columns={"oil_rate_bopd": "latest_oil_rate_bopd"})
forecast_summary = forecast_summary.merge(latest_snapshot, on=["well_id", "pad"], how="left")
forecast_summary["expected_change_pct"] = 100.0 * (
    forecast_summary["month_1_forecast_bopd"] - forecast_summary["latest_oil_rate_bopd"]
) / forecast_summary["latest_oil_rate_bopd"]

display(forecast_summary.sort_values("month_1_forecast_bopd").head(10))


In [ ]:
surveillance_df = forecast_summary.copy()
surveillance_df["driver_notes"] = ""
surveillance_df.loc[surveillance_df["water_cut_pct"] >= 45, "driver_notes"] += "High water cut; "
surveillance_df.loc[surveillance_df["uptime_pct"] <= 92, "driver_notes"] += "Low uptime; "
surveillance_df.loc[surveillance_df["recent_event_count_12m"] >= 1, "driver_notes"] += "Recent intervention history; "
surveillance_df.loc[surveillance_df["expected_change_pct"] <= -8, "driver_notes"] += "Steep near-term decline; "
surveillance_df["driver_notes"] = surveillance_df["driver_notes"].str.strip().str.rstrip(";")
surveillance_df["driver_notes"] = surveillance_df["driver_notes"].replace("", "Normal decline profile")

top_watchlist = surveillance_df.sort_values("month_1_forecast_bopd").head(5).copy()
top_watchlist = top_watchlist[[
    "well_id",
    "pad",
    "latest_oil_rate_bopd",
    "month_1_forecast_bopd",
    "avg_6m_forecast_bopd",
    "expected_change_pct",
    "driver_notes",
]]

display(top_watchlist)

plot_wells = top_watchlist["well_id"].tolist()
historical_plot_df = df.loc[df["well_id"].isin(plot_wells), ["well_id", "date", "oil_rate_bopd"]].copy()
future_plot_df = forecast_df.loc[forecast_df["well_id"].isin(plot_wells), ["well_id", "date", "forecast_oil_rate_bopd"]].copy()

fig, ax = plt.subplots(figsize=(14, 6))
for well_id in plot_wells:
    hist = historical_plot_df.loc[historical_plot_df["well_id"] == well_id]
    fut = future_plot_df.loc[future_plot_df["well_id"] == well_id]
    ax.plot(hist["date"], hist["oil_rate_bopd"], label=f"{well_id} history")
    ax.plot(fut["date"], fut["forecast_oil_rate_bopd"], linestyle="--", label=f"{well_id} forecast")

ax.set_title("History and 6-Month Forecast for Watchlist Wells")
ax.set_xlabel("Date")
ax.set_ylabel("Oil rate bopd")
ax.tick_params(axis="x", rotation=30)
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()


## Example Engineering Interpretation

Use the outputs like this during the training:

- The baseline gives a simple decline-only reference point.
- The ML model adds context from uptime, water cut, lift type, pressures, and event history.
- A well with falling forecast, high water cut, and recent downtime is a stronger surveillance candidate than a well showing only normal decline.
- This does not replace engineering judgment. It helps prioritize which wells deserve closer review first.

## Limitations and Cautions
- This is a synthetic training dataset.
- The recursive 6-month forecast assumes operating conditions stay broadly similar unless you manually change them.
- Workovers, facility constraints, choke strategy changes, and new artificial-lift settings can change outcomes materially.
- For real deployment, you would want tighter validation, scenario handling, and model monitoring.


In [ ]:
results_dir = Path.cwd() / "results" / "production_forecasting"
results_dir.mkdir(parents=True, exist_ok=True)

metrics_df.to_csv(results_dir / "model_comparison_metrics.csv", index=False)
forecast_summary.to_csv(results_dir / "well_forecast_summary.csv", index=False)
forecast_df.to_csv(results_dir / "well_forecast_6_months.csv", index=False)
top_watchlist.to_csv(results_dir / "well_watchlist.csv", index=False)

print(f"Results written to: {results_dir}")
